# Hệ thống chấm bài trắc nghiệm

Quy trình: in phiếu → học sinh tô bài → giáo viên chụp ảnh → upload đáp án và ảnh → nhận điểm.

**Cách chạy trên Colab:** chọn *Runtime → Run all*, rồi kéo xuống cuối notebook để mở giao diện. Phiếu trắng và dữ liệu mẫu nằm trong thanh *Files* bên trái.

In [ ]:
%pip install -q gradio==6.31.0

## Bước 1. Bố cục phiếu
Mọi tọa độ nằm ở một chỗ, dùng chung khi vẽ và khi đọc phiếu. Phiếu A4 ở 200 DPI có kích thước 1654×2339 px. Nhờ vậy, sau khi nắn ảnh về đúng kích thước này, ta biết chính xác vị trí từng ô.

In [ ]:
"""Toa do cua phieu tra loi, dung chung cho viec ve phieu va doc phieu.

Y tuong cot loi cua he thong: phieu la bieu mau CO DINH. Neu dua duoc anh chup ve dung khung
PAGE_W x PAGE_H thi o tron cua cau q, lua chon c luon nam o mot toa do biet truoc, khong can
"tim" o tron. File nay la nguon su that duy nhat: omr_sheet.py (ve phieu) va omr_read.py (doc
phieu) cung goi question_bubble/id_bubble, nen phieu in ra va phieu doc vao khong the lech nhau.
Doi bo cuc chi can sua o day.

He toa do: goc tren-trai la (0, 0), x sang phai, y xuong duoi, don vi pixel.
"""

# A4 = 210 x 297 mm. O 200 DPI: 210/25.4*200 = 1654, 297/25.4*200 = 2339.
# 200 DPI du net de o tron (duong kinh 40 px) con ro sau khi nan, ma anh khong qua nang.
PAGE_W = 1654
PAGE_H = 2339
NUM_QUESTIONS = 40
CHOICES = "ABCD"
ID_DIGITS = 6
BUBBLE_RADIUS = 20  # ban kinh o tron in tren phieu (px); cac vung do trong omr_read.py tinh theo so nay
MARKER_SIZE = 60  # canh o vuong dinh vi (px)
# Tam 4 o dinh vi theo thu tu vong quanh: TL, TR, BR, BL. Cach mep 80 px de o khong bi cat
# khi chup lech. Thu tu nay la thu tu getPerspectiveTransform ghep cap diem (omr_locate.warp).
MARKER_CENTERS = [(80, 80), (1574, 80), (1574, 2259), (80, 2259)]
# x, y, w, h cua thanh den dinh huong. Chi co o mep TREN, khong doi xung: 4 o goc giong het nhau
# nen phieu quay 180 do nhin y het phieu thuong; thanh den la thu duy nhat phan biet duoc.
ORIENTATION_BAR = (677, 70, 300, 20)

# Luoi ma hoc sinh: 6 cot (moi chu so mot cot), moi cot 10 o ung voi 0..9 chay doc xuong.
_ID_X, _ID_Y, _ID_DX, _ID_DY = 1050, 260, 70, 62  # o dau tien, buoc ngang giua cac cot, buoc doc giua cac so
# Luoi cau hoi: 4 cot x 10 cau; trong moi cau, A..D xep ngang cach nhau _Q_DX.
_Q_X, _Q_Y, _Q_DX, _Q_DY = 250, 1000, 70, 110
_Q_COL_GAP = 350  # khoang cach giua hai cot cau (cau 1-10, 11-20, 21-30, 31-40)
_Q_PER_COL = 10


def question_bubble(q, choice):
    """Tam o tron cua cau q (1..40), lua chon choice (A..D)."""
    # divmod(q-1, 10): cau 1..10 -> cot 0, cau 11..20 -> cot 1, ... ; row la hang trong cot
    col, row = divmod(q - 1, _Q_PER_COL)
    return (_Q_X + col * _Q_COL_GAP + CHOICES.index(choice) * _Q_DX, _Q_Y + row * _Q_DY)


def id_bubble(pos, digit):
    """Tam o tron cua chu so thu pos (0..5) trong ma hoc sinh, gia tri digit (0..9)."""
    return (_ID_X + pos * _ID_DX, _ID_Y + digit * _ID_DY)

## Bước 2. Vẽ phiếu và xuất PDF
Phiếu có 4 ô vuông đen ở góc để định vị và một thanh đen chỉ ở mép trên. Bốn ô góc giống nhau nên không phân biệt được ảnh ngược đầu; thanh đen giải quyết việc đó.

In [ ]:
"""Ve phieu tra loi va xuat PDF de in."""
import unicodedata
from functools import lru_cache

import cv2
import numpy as np
from PIL import Image, ImageDraw, ImageFont


_FONTS = ["DejaVuSans.ttf", "LiberationSans-Regular.ttf", "Arial.ttf", "Arial Unicode.ttf"]


@lru_cache(maxsize=None)
def _load_font(size):
    """Tra ve (font, co_dau): may khong co font tieng Viet thi in khong dau."""
    for name in _FONTS:
        try:
            return ImageFont.truetype(name, size), True
        except OSError:
            continue
    return ImageFont.load_default(size), False


def _ascii(text):
    text = text.replace("Đ", "D").replace("đ", "d")
    return unicodedata.normalize("NFD", text).encode("ascii", "ignore").decode()


def render_sheet():
    """Ve phieu trang, tra ve anh xam uint8 kich thuoc PAGE_H x PAGE_W."""
    img = np.full((PAGE_H, PAGE_W), 255, np.uint8)
    half = MARKER_SIZE // 2
    for x, y in MARKER_CENTERS:
        img[y - half:y + half, x - half:x + half] = 0
    bx, by, bw, bh = ORIENTATION_BAR
    img[by:by + bh, bx:bx + bw] = 0

    pts = [question_bubble(q, c) for q in range(1, NUM_QUESTIONS + 1) for c in CHOICES]
    pts += [id_bubble(p, d) for p in range(ID_DIGITS) for d in range(10)]
    for pt in pts:
        cv2.circle(img, pt, BUBBLE_RADIUS, 70, 2, cv2.LINE_AA)

    pil = Image.fromarray(img)
    draw = ImageDraw.Draw(pil)

    def text(xy, s, size=26, anchor="mm"):
        font, has_vn = _load_font(size)
        draw.text(xy, s if has_vn else _ascii(s), font=font, fill=0, anchor=anchor)

    text((180, 170), "PHIẾU TRẢ LỜI TRẮC NGHIỆM", 40, "lm")
    text((180, 300), "Họ và tên: ____________________________", 30, "lm")
    text((180, 400), "Lớp: ______________", 30, "lm")
    text((180, 520), "Tô kín một ô cho mỗi câu, dùng bút chì 2B", 26, "lm")
    text((180, 565), "hoặc bút mực tối màu.", 26, "lm")
    text((id_bubble(0, 0)[0] + 175, 195), "MÃ HỌC SINH", 30)
    for d in range(10):
        text((id_bubble(0, d)[0] - 50, id_bubble(0, d)[1]), str(d))
    for q in range(1, NUM_QUESTIONS + 1):
        x, y = question_bubble(q, "A")
        text((x - 55, y), str(q))
        if (q - 1) % 10 == 0:
            for c in CHOICES:
                cx, cy = question_bubble(q, c)
                text((cx, cy - 50), c)
    return np.array(pil)


def save_pdf(path):
    Image.fromarray(render_sheet()).save(path, "PDF", resolution=200)

## Bước 3. Tìm phiếu và nắn ảnh
1. **Ảnh xám** và **lọc Gauss** để giảm nhiễu.
2. **Phân ngưỡng thích nghi**: mỗi điểm được so với vùng lân cận, nên ánh sáng không đều vẫn tách được nét đen.
3. **Tìm contour**: giữ các hình 4 đỉnh gần vuông, đặc và tối hơn hẳn giấy bao quanh, đó là ứng viên ô định vị. Bộ 4 ô đúng phải xếp đúng tỉ lệ phiếu và khớp với thanh định hướng, nhờ vậy loại được vật đen vuông ngoài phiếu.
4. **Biến đổi phối cảnh**: từ 4 cặp điểm tính ma trận 3×3, đưa ảnh nghiêng về khung chuẩn.

In [ ]:
"""Tim phieu trong anh chup va nan ve khung chuan.

Luong chinh (buoc 1-4 cua slide): load_image -> preprocess (xam, Gauss, nguong thich nghi)
-> find_markers (tim 4 o dinh vi) -> warp (phoi canh) -> fix_orientation (sua nguoc dau).
Moi loi khong cuu duoc (khong thay 4 o, 2 phieu, khong ro chieu) deu ne SheetUnreadable de
omr_read.read_sheet doi thanh "Can xem lai" thay vi doan.
"""
from itertools import combinations

import cv2
import numpy as np
from PIL import Image, ImageOps


MAX_SIDE = 3000  # thu anh ve toi da 3000 px canh dai: du net cho o dinh vi, tranh anh 12MP lam cham
# Chi giu 12 ung vien lon nhat. find_markers thu MOI to hop 4 trong so nay: C(12,4)=495 to hop,
# con 20 ung vien la 4845. 12 du cho 4 o that + vai o gia (chu in, o tron to dam), ma van nhanh.
MAX_CANDIDATES = 12
MAX_INK_RATIO = 0.3  # (nguong "toi hon han": trung vi muc xam trong o < 0.3 x trung vi giay quanh) o dinh vi la muc den in tren giay trang: trong toi hon han vien quanh
BAR_SLACK = 40  # px: giay cong lam thanh dinh huong lech khoi vi tri chuan sau khi nan
_MARKER_RATIO = (MARKER_CENTERS[2][1] - MARKER_CENTERS[1][1]) / (MARKER_CENTERS[1][0] - MARKER_CENTERS[0][0])


class SheetUnreadable(Exception):
    def __init__(self, reason):
        super().__init__(reason)
        self.reason = reason


def load_image(path):
    """Doc anh, ap dung huong EXIF cua dien thoai, gioi han kich thuoc; tra ve BGR."""
    try:
        with Image.open(path) as im:
            rgb = np.array(ImageOps.exif_transpose(im).convert("RGB"))
        scale = MAX_SIDE / max(rgb.shape[:2])
        if scale < 1:
            rgb = cv2.resize(rgb, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)
    except Exception as e:
        raise SheetUnreadable("Không đọc được file ảnh") from e
    return cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)


def preprocess(bgr):
    """Anh xam -> loc nhieu -> phan nguong thich nghi (block lon hon duong kinh o tron)."""
    # Xam: vi muc chi can do "toi/sang", mau khong them thong tin, ma giam 3 lan du lieu.
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    # Gauss 5x5: lam min nhieu hat/JPEG de contour khong bi rang cua.
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    # Nguong thich nghi: moi diem duoc so voi trung binh Gauss cua 101x101 diem quanh no (tru C=10).
    # Nguong toan cuc (Otsu) hong khi anh co ben toi ben sang; cach nay chi quan tam "toi hon
    # lan can", nen chiu duoc anh sang khong deu. Block 101 phai lon hon o dinh vi de trung binh
    # lan can chu yeu la giay (neu block nho hon o, giua o dac se "bang" nen khong con la toi hon
    # lan can). THRESH_BINARY_INV: net den -> 255
    # (trang) de findContours coi vat the la tien canh.
    binary = cv2.adaptiveThreshold(blurred, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                   cv2.THRESH_BINARY_INV, 101, 10)
    return gray, blurred, binary


def _ink_on_paper(gray, contour, area):
    """Trong contour toi hon han dai giay bao quanh; loai phim ban phim, vat toi tren nen toi.

    Chi xet khung bao cua contour noi rong them nua cua so gian no, khong tao mat na ca anh.
    """
    k = max(int(np.sqrt(area) * 0.25), 3) | 1  # be rong vanh giay = 1/4 canh o, ep le (| 1) cho kernel
    x, y, w, h = cv2.boundingRect(contour)
    x0, y0 = max(x - k // 2, 0), max(y - k // 2, 0)
    roi = gray[y0:y + h + k // 2, x0:x + w + k // 2]
    mask = np.zeros(roi.shape, np.uint8)
    cv2.drawContours(mask, [contour], -1, 255, -1, offset=(-x0, -y0))
    # vanh = ban gian no tru ban goc -> dai giay ngay sat ngoai contour
    ring = cv2.dilate(mask, np.ones((k, k), np.uint8)) - mask
    # So trung vi (khong phai trung binh) de it bi anh huong boi vai diem sang/toi bat thuong.
    return np.median(roi[mask > 0]) < MAX_INK_RATIO * np.median(roi[ring > 0])


def _candidates(binary, gray):
    """Contour gan vuong, dac, lon, la muc tren giay: ung vien o dinh vi. Tra ve [(dien_tich, tam)]."""
    # RETR_LIST: lay moi contour, khong lap cay long nhau (o dinh vi la khoi dac, khong can phan cap)
    contours, _ = cv2.findContours(binary, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)
    # Dien tich toi thieu = (canh dai anh / 200)^2, tuc canh ~0.5% anh: chi de loai hat nhieu li ti.
    # Muc nay rat thap so voi o that (~2.5% chieu cao phieu), nen khong bo sot o ke ca khi phieu
    # chiem it anh; cac bo loc ben duoi moi la thu loai o gia.
    min_area = (max(binary.shape) / 200) ** 2
    found = []
    for c in contours:
        area = cv2.contourArea(c)
        if area < min_area or area > 0.02 * binary.size:  # qua nho = nhieu, qua lon (>2% anh) = khong phai o
            continue
        # Xap xi da giac, sai so 6% chu vi: o vuong that ra dung 4 dinh, chu cai/o tron thi khong.
        approx = cv2.approxPolyDP(c, 0.06 * cv2.arcLength(c, True), True)
        if len(approx) != 4 or not cv2.isContourConvex(approx):
            continue
        (cx, cy), (w, h), _ = cv2.minAreaRect(c)  # hop chu nhat nho nhat, chiu duoc xoay
        # phoi canh manh lam o vuong thanh hinh binh hanh, nen do dac tinh theo bao loi
        if min(w, h) == 0 or not 0.5 < w / h < 2.0 or area / cv2.contourArea(cv2.convexHull(c)) < 0.9:
            continue
        if not _ink_on_paper(gray, c, area):
            continue
        # vien trong va vien ngoai cua cung mot o cho hai contour trung tam: chi giu mot
        # (nguong 10 px: hai contour cua cung o co tam lech nhau chi vai px)
        if all(np.hypot(cx - fx, cy - fy) > 10 for _, (fx, fy) in found):
            found.append((area, (cx, cy)))
    return sorted(found, key=lambda f: -f[0])[:MAX_CANDIDATES]


def _order(pts):
    """Sap xep 4 goc theo vong, canh dai cua tu giac la chieu doc cua phieu."""
    # Sap theo goc quanh tam: 4 diem cua mot tu giac loi se di vong tron, bat ke phieu xoay the nao.
    center = pts.mean(axis=0)
    angles = np.arctan2(pts[:, 1] - center[1], pts[:, 0] - center[0])
    p = pts[np.argsort(angles)]
    # Phieu A4 dung (cao > rong): canh p0-p1 phai la canh NGAN (canh ngang tren). Neu no dai hon
    # canh ke p1-p2 thi ta dang bat dau o cung sai -> xoay vong 1 vi tri. Chua biet tren/duoi:
    # viec do (xoay 180 do) la cua fix_orientation.
    if np.linalg.norm(p[0] - p[1]) > np.linalg.norm(p[1] - p[2]):
        p = np.roll(p, -1, axis=0)
    return p


def _rect_error(p):
    """Do lech cua tu giac so voi hinh chu nhat ti le nhu phieu; None neu khong hop le."""
    if not cv2.isContourConvex(p.astype(np.float32)):
        return None
    top, right, bottom, left = (np.linalg.norm(p[(i + 1) % 4] - p[i]) for i in range(4))
    if min(top, bottom) < 100:  # qua nho: nhieu hoac cac o gia nam sat nhau
        return None
    # Phoi canh vua phai chi lam hai canh doi dien chenh nhau chut; chenh >1.5 lan la khong phai phieu.
    if max(top, bottom) / min(top, bottom) > 1.5 or max(left, right) / min(left, right) > 1.5:
        return None
    # Ti le (canh doc / canh ngang) cua phieu that la (2259-80)/(1574-80)=1.46. So voi ti le do duoc,
    # cho sai 35% de du dung cho phoi canh, nhung du chat de loai bo cac to hop vo nghia.
    ratio_err = abs((left + right) / (top + bottom) - _MARKER_RATIO) / _MARKER_RATIO
    return ratio_err if ratio_err < 0.35 else None


_BAR_X0 = ORIENTATION_BAR[0] + 20  # vung chua thanh dinh huong trong khung chuan, cho lech +-BAR_SLACK
_BAR_Y0 = ORIENTATION_BAR[1] - BAR_SLACK
_BAR_SIZE = (ORIENTATION_BAR[2] - 40, ORIENTATION_BAR[3] + 2 * BAR_SLACK)  # (rong, cao)
# toa do khung chuan cua vung o mep tren va (sau khi xoay 180 do) o mep duoi
_BAR_TOP = np.float64([[1, 0, _BAR_X0], [0, 1, _BAR_Y0], [0, 0, 1]])
_BAR_BOTTOM = np.float64([[-1, 0, PAGE_W - 1 - _BAR_X0], [0, -1, PAGE_H - 1 - _BAR_Y0], [0, 0, 1]])


def _bar_level(band, darkest):
    """Muc sang cua dai cao bang thanh dinh huong, toi nhat (hoac sang nhat) trong vung."""
    # Thanh cao 20 px. Lay trung binh theo hang (mean axis=1) -> 1 gia tri sang moi hang, roi truot
    # cua so cao 12 px (=20-8, chua 8 px du phong lech/mo): tim vi tri cua so toi (hoac sang) nhat.
    # Thanh that du dai de lap day cua so; chu in/nhieu thi khong du dai nen khong dat muc cuc tri.
    h = ORIENTATION_BAR[3] - 8
    bands = np.convolve(band.mean(axis=1), np.ones(h) / h, mode="valid")
    return bands.max() if darkest else bands.min()


def _bar_band(img):
    w, h = _BAR_SIZE
    return img[_BAR_Y0:_BAR_Y0 + h, _BAR_X0:_BAR_X0 + w]


def _bar_present(binary, corners):
    """Dung bo 4 o thi thanh dinh huong phai nam o mep tren hoac duoi.

    Chi nan vung quanh thanh dinh huong (khong nan ca phieu) vi buoc nay chay cho moi bo 4 o.
    """
    # m: khung chuan -> anh goc (chieu nguoc voi warp). Chi cat cua so quanh thanh (WARP_INVERSE_MAP
    # voi "place" dich/lat toi vung do) de khong phai nan ca phieu cho moi bo 4 o con lai sau cac bo loc.
    m = cv2.getPerspectiveTransform(np.float32(MARKER_CENTERS), np.float32(corners)).astype(np.float64)
    top, bottom = (_bar_level(cv2.warpPerspective(binary, m @ place, _BAR_SIZE,
                                                  flags=cv2.INTER_LINEAR | cv2.WARP_INVERSE_MAP), True)
                   for place in (_BAR_TOP, _BAR_BOTTOM))
    # Dung la phieu: mot dau co thanh den (>0.7 trang trong anh nhi phan) va dau kia KHONG co (<0.3).
    # Hai dau deu co hoac deu khong -> khong phai phieu cua minh (vd 4 o tron to dam ben trong phieu).
    return max(top, bottom) > 0.7 * 255 and min(top, bottom) < 0.3 * 255


def _outside(sheet, other):
    """Bo 4 o nay nam han ngoai phieu da chon: trong anh co phieu thu hai."""
    return all(cv2.pointPolygonTest(sheet, (float(x), float(y)), False) < 0 for x, y in other)


def find_markers(binary, gray):
    """Tim 4 o dinh vi: to hop phai dung ti le phieu va khop thanh dinh huong."""
    ranked = []
    for combo in combinations(_candidates(binary, gray), 4):
        areas = [c[0] for c in combo]
        # 4 o in cung kich thuoc; phoi canh mac dinh chi doi dien tich < 2 lan.
        if max(areas) / min(areas) > 2.0:
            continue
        p = _order(np.float32([c[1] for c in combo]))
        err = _rect_error(p)
        if err is not None:
            ranked.append((err, p))
    valid = [p for _, p in ranked if _bar_present(binary, p)]
    if not valid:
        raise SheetUnreadable("Không tìm thấy 4 ô định vị")
    # bo lon nhat la phieu that; cac o da to ben trong phieu cung co the xep thanh bo gia
    valid.sort(key=lambda p: -np.linalg.norm(p[0] - p[2]))
    if any(_outside(valid[0], other) for other in valid[1:]):
        raise SheetUnreadable("Ảnh có nhiều hơn một phiếu")
    return valid[0]


def warp(gray, corners):
    """Bien doi phoi canh ve khung chuan PAGE_W x PAGE_H."""
    # Ma tran homography 3x3 tinh tu dung 4 cap diem (4 goc thay trong anh -> 4 tam chuan).
    # 4 cap la so toi thieu de giai 8 an cua homography; vi vay moi o dinh vi quan trong.
    m = cv2.getPerspectiveTransform(np.float32(corners), np.float32(MARKER_CENTERS))
    return cv2.warpPerspective(gray, m, (PAGE_W, PAGE_H), flags=cv2.INTER_LINEAR)


def fix_orientation(warped):
    """Dua vao thanh den chi co o mep tren de xac dinh chieu phieu, xoay 180 do neu nguoc."""
    # Sau warp, neu phieu xuoi thi thanh den o tren. Do muc sang vung tren (top) va vung tren cua
    # anh da lat 180 do (bottom = chinh la mep duoi that). Vung co thanh den toi hon -> la mep tren.
    top = _bar_level(_bar_band(warped), False)
    flipped = cv2.rotate(warped, cv2.ROTATE_180)
    bottom = _bar_level(_bar_band(flipped), False)
    if min(top, bottom) > 0.6 * max(top, bottom):  # hai dau gan nhu nhau: khong phan biet duoc, khong doan
        raise SheetUnreadable("Không xác định được chiều phiếu")
    return warped if top < bottom else flipped

## Bước 4. Đọc ô tô
- **Bù giấy cong**: tìm viền các ô in, ghép với vị trí chuẩn, ước lượng độ lệch mềm (đa thức rồi nội suy Gauss cục bộ) và nắn lại ảnh.
- **Khử ánh sáng**: phép đóng hình thái ước lượng nền giấy, chia ảnh cho nền và cho mức sáng giấy của chính phiếu để giấy gần 1 và mực gần 0.
- **Căn lại tâm ô**: khớp mẫu vòng tròn với viền in để bù sai số còn lại sau khi nắn.
- **Tỉ lệ điểm đen** trong lõi mỗi ô (bỏ viền in): dưới 0.25 là trống, trên 0.75 là đã tô, ở giữa là không chắc. Ô tô đủ diện tích nhưng quá nhạt cũng là không chắc.
- **"Trống" phải có bằng chứng**: phần trong viền ô và giấy quanh ô đều sạch. Nếu không, ô được coi là không chắc, nhờ vậy nét tô nhạt, vết tẩy và bóng đổ không bị chấm nhầm.
- **Kiểm tra chất lượng**: độ dốc tại cạnh ô định vị cho biết ảnh mờ; không thấy viền in của nhiều ô nghĩa là ảnh lóa hoặc lưới lệch.

In [ ]:
"""Doc cac o da to tren phieu da nan va kiem tra chat luong anh.

Luong cua read_sheet (buoc 5-7 cua slide):
  1. unbend: bu giay cong tren phieu da nan phoi canh.
  2. _normalize_light + _blank_reference: khu anh sang, quy ve "giay ~ 1, muc ~ 0".
  3. check_quality: anh mo / loa / luoi lech -> dua vao problems.
  4. _read_id va doc tung cau: moi o ra mot trong 3 trang thai blank / filled / uncertain.

Triet ly thiet ke: moi cho nghi ngo deu tra "uncertain", khong bao gio doan. Mot o "trong" khong
chi can it den ma con phai CO BANG CHUNG la trong (vien in thay ro, ben trong sach, giay quanh sach).
Cac hang so duoi day deu duoc chinh tren anh mo phong (tools/evaluate.py), CHUA chinh tren anh that.
"""
from dataclasses import dataclass, field

import cv2
import numpy as np


# --- Phan loai theo ti le diem den trong loi o (slide 8) ---
BLANK_BELOW = 0.25  # ti le diem den < 0.25 -> trong
FILLED_ABOVE = 0.75  # > 0.75 -> da to; o giua 0.25-0.75 la "khong chac" (to do dang, tay xoa, ...)
# Khoang giua rong co chu y: neu chi dung mot nguong 0.5, o to nua chung se bi doan dai khai.
DARK_PIXEL = 0.75  # do sang so voi nen giay; toi hon muc nay la diem den
SOLID_MEAN = 0.65  # o da to phai du dam; to nhat la "khong chac"
# "O trong" duoc so voi o trong dien hinh cua chinh phieu (trung vi cua moi o): do mo, vien in day,
# nhieu hat giay cua anh chup co san trong mau nen tu triet tieu. Gia tri la muc toi hon mau.
# Cac nguong *_DEV la do TOI HON mau o trong (blank_ref - patch), tinh tren anh da chuan hoa
# (giay ~ 1). 0.08 nghia la toi hon o trong dien hinh khoang 8% muc giay.
INNER_DEV = 0.08  # trung binh phan trong vien: vet tay do, net to nhat
OUTSIDE_DEV = 0.07  # giay quanh o: bong do, vat den
INK_DEV = 0.25  # diem muc dam
MAX_INNER_INK = 0.02  # net to nho, vien o to bi loa
FAINT_DEV = 0.07  # diem hoi toi (sau khi lam min de bo nhieu hat): net chi nhat
MAX_INNER_FAINT = 0.08  # o trong that toi da ~0.06, net nhat rat mo tren anh mo ~0.10
MAX_DARKNESS_GAP = 0.15  # hai o cung to phai dam nhu nhau, khac nhieu la vet tay
MAX_MISSING_RINGS = 0.05  # qua 5% o khong thay vien in -> ca anh bi loa hoac luoi lech
MIN_SHARPNESS = 0.07  # mo hon muc nay thi net to nhat co the bi nhoe mat; mo nhe van doc dung
RING_COVERAGE = 0.7  # vien duoc coi la "thay" neu >= 70% trong 36 huong con ro (xem _ring_visible)
RING_CONTRAST = 0.5  # vien thay duoc khi con it nhat ngan nay do tuong phan cua vien o mau
STEP_MAX_SIDE = 1000  # chi thu nho anh luu cho giao dien (r.steps); khong anh huong ket qua doc


@dataclass
class SheetReading:
    student_id: str | None = None
    id_problems: list = field(default_factory=list)
    marks: dict = field(default_factory=dict)
    uncertain: list = field(default_factory=list)
    problems: list = field(default_factory=list)
    steps: dict = field(default_factory=dict)
    warped: np.ndarray | None = None


def classify_ratio(ratio):
    """Ti le diem den (0..1) -> "blank" / "filled" / "uncertain". Khong co nhanh nao tra ve doan."""
    if ratio < BLANK_BELOW:
        return "blank"
    return "filled" if ratio > FILLED_ABOVE else "uncertain"


def _normalize_light(warped):
    """Loc Gauss roi chia cho nen giay uoc luong duoc de khu anh sang khong deu; giay ~ 1, muc ~ 0."""
    blurred = cv2.GaussianBlur(warped, (5, 5), 0)
    # MORPH_CLOSE = dan no roi co lai: dan no (lay max) xoa cac vet TOI hep hon kernel (net mut, chu,
    # o tron in), co lai tra lai hinh dang vung sang. Ket qua = "nen giay" khong co muc, ke ca cho
    # toi vi sang khong deu. Kernel elip 61 px phai LON hon net mut/duong kinh o (40 px) de xoa het
    # muc, nhung van nho de theo kip bong do thay doi cham.
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (61, 61))
    background = cv2.morphologyEx(blurred, cv2.MORPH_CLOSE, kernel)
    # Chia anh cho nen (khong phai tru): giay ~ 1 bat ke sang hay toi, muc ~ 0. Phep chia triet tieu
    # dung hang so nhan cua anh sang (anh sang = he so nhan len do phan xa cua giay). max(.,1) tranh chia 0.
    return blurred.astype(np.float32) / np.maximum(background.astype(np.float32), 1)


# Cac "mat na" tren patch (anh cat quanh mot o), tinh mot lan:
#   _CORE    loi o, ban kinh 0.7R=14 px: bo vien in de vien khong bi tinh la "muc", va chiu duoc
#            tam o lech vai px.
#   _INNER   long trong vien in (R-5): noi nghi ngo vet tay/net nhat.
#   _OUTSIDE vanh giay ngoai o (R+4..R+10): neu toi hon giay chung -> bong do hoac vat den de len.
_R = int(BUBBLE_RADIUS * 0.7)
_P = BUBBLE_RADIUS + 10  # nua canh patch, chua het vanh ngoai
_YY, _XX = np.mgrid[-_P:_P + 1, -_P:_P + 1]
_DIST = np.hypot(_XX, _YY)
_CORE = _DIST <= _R
_INNER = _DIST <= BUBBLE_RADIUS - 5  # chua 1 px cho cho giay cong lam vien o nho lai
_FAINT_ZONE = _DIST <= BUBBLE_RADIUS - 7  # net nhat chi xet xa vien, vien lech nhe khong lam nham
_OUTSIDE = (_DIST >= BUBBLE_RADIUS + 4) & (_DIST <= _P)
_ANGLES = np.arange(36) * 2 * np.pi / 36


def _rays(radii):
    """Toa do lay mau trong patch theo 36 huong. Nhieu diem roi dung vao x.5: lam tron len co
    chu dong (+1e-9) de sai so bit cuoi khong doi diem lay mau, giong het ban MATLAB."""
    pts = [[(_P + r * np.sin(a), _P + r * np.cos(a)) for r in radii] for a in _ANGLES]
    return np.floor(np.array(pts) + 0.5 + 1e-9).astype(int)


# vung vien (R-4..R+4) va giay ngay ben ngoai (ben trong co the da to kin)
_RAYS_RING = _rays(range(BUBBLE_RADIUS - 4, BUBBLE_RADIUS + 5))
_RAYS_PAPER = _rays((BUBBLE_RADIUS + 8, BUBBLE_RADIUS + 9))


_SNAP = 12  # px: sai so nan cuc bo toi da duoc bu lai; nho hon nhieu so voi khoang cach hai o
_T = BUBBLE_RADIUS + 9


def _ring_template():
    """Vong duong o vien in, vong am ngay ben ngoai: o da to cung chi khop dung o mot vi tri.

    Hai vong deu chuan hoa tong = 1, nen diem khop = do toi cua vien tru do toi cua giay quanh.
    """
    yy, xx = np.mgrid[-_T:_T + 1, -_T:_T + 1]
    dist = np.hypot(xx, yy)
    pos = (np.abs(dist - BUBBLE_RADIUS) <= 2.5).astype(np.float32)  # vong day 5 px
    neg = (np.abs(dist - BUBBLE_RADIUS - 6) <= 2.5).astype(np.float32)
    return pos / pos.sum() - neg / neg.sum()


_RING_TEMPLATE = _ring_template()


def ring_score(light):
    """Ban do diem khop vien o tren anh da khu sang (khong qua nhi phan nen chiu duoc anh mo).

    score[y - _T, x - _T] la diem cua o co tam (x, y).
    """
    # 1 - light: dao de vien in (toi) thanh gia tri cao. TM_CCORR (tuong quan khong chuan hoa) vi mau
    # da tong = 1 va co thanh phan am, nen diem = (do toi trung binh cua vong vien) - (cua vong giay ngoai).
    return cv2.matchTemplate(1 - light, _RING_TEMPLATE, cv2.TM_CCORR)


def _snap(score, center):
    """Can lai tam o theo vien in gan nhat, bu sai so phoi canh va giay cong."""
    # Tim diem khop cao nhat trong cua so +-12 px quanh tam ly thuyet. Day la "buoc 7: can tam
    # theo vien in". 12 px la nho so voi khoang cach 2 o gan nhat (~70 px) nen khong nhay sang o ben canh.
    x, y = center
    window = score[y - _T - _SNAP:y - _T + _SNAP + 1, x - _T - _SNAP:x - _T + _SNAP + 1]
    _, best, _, (dx, dy) = cv2.minMaxLoc(window)
    # Khong co khop duong (anh toi den / vien bi che hoan toan) -> giu nguyen toa do ly thuyet.
    return (x, y) if best <= 0 else (x + dx - _SNAP, y + dy - _SNAP)


# --- Bu giay cong (buoc 5): uoc luong truong lech giua o chuan va o that ---
# Y tuong: phieu da nan phoi canh phang, nhung giay cong lam o that lech vai chuc px khoi o chuan.
# Tim cac vien o that (dinh cua ring_score), ghep tung o chuan voi dinh gan nhat -> thu duoc cac
# vector lech. Khop tu tho den min: da thuc bac thap (hinh dang tong the, it tu do, kho "hoc vet")
# roi tang bac, cuoi cung noi suy Gauss cuc bo cho cho giay gat. Neu khong tin cay -> khong sua.
_PEAK_COUNT = 2  # giu so dinh manh nhat gap doi so o; dinh thua (chu, vet ban) bi loai khi ghep
_POLY_STEPS = ((1, 45), (2, 25), (3, 15))  # (bac da thuc, ban kinh ghep o) tu tho den min
_LOCAL_STEPS = (30, 20, 12, 8)  # ban kinh ghep o khi bu cong cuc bo (vung gan tay cam)
_LOCAL_SIGMA = 60  # px: tam anh huong cua mot o len do lech cua vung quanh no
_MIN_MATCHED = 0.5
_MAX_BEND = 80  # px: lech lon hon muc nay thi khong tin la giay cong
_ANCHOR_WEIGHT = 5


def _ring_peaks(score):
    """Tam cac vien o tim thay tren toan phieu da nan, manh nhat truoc."""
    # Cuc dai cuc bo: diem bang max trong cua so 21x21 (dilate = max filter) va co diem duong.
    peaks = (score >= cv2.dilate(score, np.ones((21, 21), np.uint8))) & (score > 0)
    ys, xs = np.nonzero(peaks)
    keep = np.argsort(-score[ys, xs])[:_PEAK_COUNT * len(_all_bubbles())]
    return np.column_stack([xs[keep] + _T, ys[keep] + _T]).astype(np.float32)


def _poly(pts, deg):
    # Chuan hoa toa do ve 0..1 de cac so hang u^i v^j cung cap, he phuong trinh khong bi suy bien.
    # Cac cot la cac don thuc u^i v^j voi i+j <= deg (da thuc 2 bien).
    u, v = pts[:, 0] / PAGE_W, pts[:, 1] / PAGE_H
    return np.column_stack([u ** i * v ** j for i in range(deg + 1) for j in range(deg + 1 - i)])


def _smooth(src, values, pts):
    """Noi suy Gauss cac gia tri tai src ve pts; xa moi o thi tat dan ve 0."""
    d2 = ((pts[:, None] - src[None]) ** 2).sum(axis=2)
    # Trong so Gauss theo khoang cach (sigma 60 px). +0.05 o mau so: xa moi o thi tong trong so ~0,
    # ket qua tat dan ve 0 (khong sua gi) thay vi keo ra gia tri vo ly.
    w = np.exp(-d2 / (2 * _LOCAL_SIGMA ** 2))
    return (w @ values) / (w.sum(axis=1) + 0.05)[:, None]


def _bend_field(score):
    """Uoc luong do lech giua vi tri o chuan va vi tri o that; None neu khong tin cay.

    Tra ve ham tinh do lech tai cac diem bat ky. Ghep o chuan voi vien o gan nhat, tu ban kinh
    lon va da thuc bac thap toi bac cao, roi bu phan du cuc bo cho cho giay cong gat.
    O dinh vi da duoc nan dung nen co do lech 0.
    """
    template = np.float32(_all_bubbles())
    peaks = _ring_peaks(score)
    if len(peaks) < 2:
        return None

    def match(pred, radius):
        # Ghep o chuan (da cong do lech du doan) voi vien that gan nhat. Chi nhan neu:
        #   d1 < radius       : du gan;
        #   d2 > 1.5 * d1     : dinh gan nhat ro hon dinh gan nhi 1.5 lan -> khong nhap nhang giua hai o.
        # Neu khong thoa thi bo o do (ok=False), khong ep ghep.
        dist = np.linalg.norm(pred[:, None] - peaks[None], axis=2)
        order = np.argsort(dist, axis=1)[:, :2]
        d1, d2 = np.take_along_axis(dist, order, axis=1).T
        ok = (d1 < radius) & (d2 > 1.5 * d1)
        return ok, peaks[order[:, 0]] - template

    # 4 o dinh vi da dung toa do (do chinh la thu nan ve) nen ep lech = 0 tai do, lap 5 lan de co trong so cao.
    anchors = np.repeat(np.float32(MARKER_CENTERS), _ANCHOR_WEIGHT, axis=0)
    disp = np.zeros_like(template)
    for deg, radius in _POLY_STEPS:
        ok, target = match(template + disp, radius)
        if ok.mean() < _MIN_MATCHED:  # ghep duoc < 50% o thi khong dang tin: ngat, khong sua anh
            return None
        src = np.vstack([template[ok], anchors])
        dst = np.vstack([target[ok], np.zeros_like(anchors)])
        # Binh phuong toi thieu: tim he so da thuc sao cho do lech du doan ~ do lech do duoc
        coef = np.linalg.lstsq(_poly(src, deg), dst, rcond=None)[0]
        disp = _poly(template, deg) @ coef

    def base(pts):
        return _poly(pts, deg) @ coef

    # Phan du (do lech con lai sau da thuc) duoc noi suy Gauss cuc bo, ban kinh ghep thu hep 30 -> 8 px.
    for radius in _LOCAL_STEPS:
        ok, target = match(template + disp, radius)
        if ok.mean() < _MIN_MATCHED:
            return None
        src, residual = template[ok], target[ok] - base(template[ok])
        disp = base(template) + _smooth(src, residual, template)
    return lambda pts: base(pts) + _smooth(src, residual, pts)


# Tinh truong lech tren luoi thua 16 px (rat re) roi noi suy tuyen tinh ra tung diem anh (de remap).
# Tinh truc tiep tren ~3.9 trieu diem se cham hon rat nhieu ma do lech doi cham nen khong mat chinh xac.
_GRID_STEP = 16


def _lerp_matrix(n, count):
    """Ma tran noi suy tuyen tinh tu luoi 0, step, 2*step, ... (count diem) ra cac toa do 0..n-1."""
    pos = np.arange(n) / _GRID_STEP
    i = np.minimum(pos.astype(int), count - 2)
    t = pos - i
    m = np.zeros((n, count), np.float32)
    m[np.arange(n), i] = 1 - t
    m[np.arange(n), i + 1] = t
    return m


def unbend(warped):
    """Bu giay cong: nan lai anh de cac o that trung voi vi tri chuan; khong tin cay thi giu nguyen."""
    field = _bend_field(ring_score(_normalize_light(warped)))
    if field is None:
        return warped
    gy, gx = np.mgrid[0:PAGE_H + _GRID_STEP:_GRID_STEP, 0:PAGE_W + _GRID_STEP:_GRID_STEP]
    disp = field(np.column_stack([gx.ravel(), gy.ravel()]).astype(np.float32))
    # Lech > 80 px khong the la giay cong nhe (o cach nhau ~70 px): nhieu kha nang ghep nham o. Giu
    # nguyen anh de khong lam hong them; check_quality se bat neu luoi that su lech.
    if np.abs(disp).max() > _MAX_BEND:
        return warped
    my, mx = _lerp_matrix(PAGE_H, gx.shape[0]), _lerp_matrix(PAGE_W, gx.shape[1])
    dx = my @ disp[:, 0].reshape(gx.shape).astype(np.float32) @ mx.T
    dy = my @ disp[:, 1].reshape(gx.shape).astype(np.float32) @ mx.T
    xs, ys = np.meshgrid(np.arange(PAGE_W, dtype=np.float32), np.arange(PAGE_H, dtype=np.float32))
    # remap lay pixel tai (x+dx, y+dy): keo o that ve dung vi tri chuan. BORDER_REPLICATE tranh vien den.
    return cv2.remap(warped, xs + dx, ys + dy, cv2.INTER_LINEAR, borderMode=cv2.BORDER_REPLICATE)


def _patch(norm, center):
    x, y = center
    return norm[y - _P:y + _P + 1, x - _P:x + _P + 1]


def _bubble_state(norm, center, blank_ref):
    """Ty le diem den trong loi o -> filled / blank / uncertain.

    "Trong" can bang chung: ben trong vien in va giay xung quanh giong o trong dien hinh cua phieu.
    """
    patch = _patch(norm, center)
    core = patch[_CORE]
    dev = blank_ref - patch  # > 0 nghia la toi hon o trong dien hinh
    # (1) Giay quanh o toi hon binh thuong -> bong do / vat den de len: khong tin duoc ca o.
    if dev[_OUTSIDE].mean() > OUTSIDE_DEV:
        return "uncertain"
    # (2) Dem ti le diem den trong loi o roi phan loai (slide 8).
    state = classify_ratio(float((core < DARK_PIXEL).mean()))
    if state == "filled":
        # Nhieu diem toi nhung chua DU DAM (trung binh > 0.65) -> net nhat hoac to hoi, khong chac.
        return "filled" if core.mean() <= SOLID_MEAN else "uncertain"
    # (3) blank / uncertain: tim dau hieu "khong sach" ben trong -> chuyen thanh uncertain.
    #     Vet tay do (trung binh vung trong), vai diem muc dam (net to nho / vien bi loa),
    #     hoac net nhat dien rong sau khi lam min 7x7 (de bo hat nhieu). Day la cac ca "tay chua sach".
    inner = dev[_INNER]
    faint = cv2.blur(dev.astype(np.float32), (7, 7))[_FAINT_ZONE]
    if (inner.mean() > INNER_DEV or (inner > INK_DEV).mean() > MAX_INNER_INK
            or (faint > FAINT_DEV).mean() > MAX_INNER_FAINT):
        return "uncertain"
    return state


def _blank_reference(norm, score):
    """O trong dien hinh cua phieu: trung vi patch cua moi o (da so o la o trong).

    Chia ca anh va mau cho muc giay cua mau de giay ~ 1 bat ke giay xam hay anh toi.
    """
    # Trung vi theo tung pixel tren 220 patch (40x4 o cau hoi + 6x10 o ma): da so o la o trong nen o to bi loai ra, chi con
    # "o trong dien hinh CUA PHIEU NAY" (da gom mo, do day vien in, hat giay cua chinh anh chup).
    ref = np.median([_patch(norm, _snap(score, p)) for p in _all_bubbles()], axis=0)
    # level = muc giay thuc trong mau; chia de giay = 1 ke ca khi giay xam. max(.,0.5) chong chia gan 0.
    level = max(float(ref[_OUTSIDE].mean()), 0.5)
    return norm / level, ref / level


def _ring_contrast(patch):
    """Do tuong phan vien in theo 36 huong: giay ben ngoai tru diem toi nhat trong vung vien."""
    # Theo 36 huong tu tam: trong dai vien (R-4..R+4) lay diem TOI NHAT (vien in), tru muc giay
    # ngay ngoai. Hieu so lon = vien ro tren huong do; mot vet muc to kin se lam mat tuong phan.
    ring = patch[_RAYS_RING[..., 0], _RAYS_RING[..., 1]].min(axis=1)
    paper = patch[_RAYS_PAPER[..., 0], _RAYS_PAPER[..., 1]].mean(axis=1)
    return paper - ring


def _ring_visible(norm, center, blank_ref):
    """Vien in cua o co ro nhu o trong dien hinh khong? Bi loa che thi mat vien o nhieu huong."""
    seen = _ring_contrast(_patch(norm, center)) >= RING_CONTRAST * _ring_contrast(blank_ref)
    return seen.mean() >= RING_COVERAGE


def _all_bubbles():
    pts = [question_bubble(q, c) for q in range(1, NUM_QUESTIONS + 1) for c in CHOICES]
    return pts + [id_bubble(p, d) for p in range(ID_DIGITS) for d in range(10)]


def _edge_sharpness(warped):
    """Do doc lon nhat tai canh trai cua cac o dinh vi, chia cho do tuong phan; canh net ~ 0.5."""
    # Mo -> canh o vuong den (chuyen toi/sang) chuyen tu tu. Do "do doc lon nhat" cua profile ngang
    # qua canh trai, chia cho tuong phan (max-min) de khong phu thuoc anh sang. Canh net ~0.5,
    # mo nhieu -> nho. Lay MIN trong 4 o: chi can mot goc mo la anh mo.
    half = MARKER_SIZE // 2
    values = []
    for x, y in MARKER_CENTERS:
        profile = warped[y - 15:y + 15, x - half - 25:x - half + 25].mean(axis=0)
        contrast = float(profile.max() - profile.min())
        values.append(float(np.abs(np.diff(profile)).max()) / max(contrast, 1.0))
    return min(values)


def check_quality(warped, norm, score, blank_ref):
    """Bao anh mo qua do net cua canh; bao loa/lech khi khong thay vien cua nhieu o."""
    problems = []
    if _edge_sharpness(warped) < MIN_SHARPNESS:
        problems.append("Ảnh bị mờ")
    pts = _all_bubbles()
    # Dem so o khong thay vien in. Loa trang xoa mat vien, luoi lech thi vien nam sai cho; ca hai
    # lam > 5% o mat vien -> dung lai, khong chap nhan ket qua doc.
    missing = sum(not _ring_visible(norm, _snap(score, p), blank_ref) for p in pts)
    if missing / len(pts) > MAX_MISSING_RINGS:
        problems.append("Ảnh bị lóa hoặc lưới ô bị lệch")
    return problems


def _read_id(norm, score, blank_ref):
    digits, problems = [], []
    for pos in range(ID_DIGITS):
        states = [_state(norm, score, id_bubble(pos, d), blank_ref) for d in range(10)]
        # Moi cot chu so can DUNG MOT o to va khong o nao uncertain; con lai (khong to, to 2 so,
        # hoac nghi ngo) deu khong doan ma - mot chu so sai la sai ca ma hoc sinh.
        filled = [d for d, s in enumerate(states) if s == "filled"]
        if "uncertain" in states or len(filled) != 1:
            problems.append(f"Mã học sinh: chữ số thứ {pos + 1} không đọc được")
        else:
            digits.append(str(filled[0]))
    return ("".join(digits) if not problems else None), problems


def _state(norm, score, center, blank_ref):
    """O trong ma khong thay vien (bi loa che) thi khong the coi la chac chan trong."""
    center = _snap(score, center)  # can tam theo vien in truoc khi do
    state = _bubble_state(norm, center, blank_ref)
    if state == "blank" and not _ring_visible(norm, center, blank_ref):
        return "uncertain"
    return state


def _uneven_marks(norm, score, q, filled):
    """Hai o to dam khac han nhau: o nhat hon co the la vet tay chua sach."""
    # Tinh co hieu khi >= 2 o cung to (tuc cau "to nhieu o"). Neu mot o den han, o kia nhat hon
    # han (chenh > 0.15) thi nhieu kha nang o nhat la vet tay chua tay sach, KHONG phai y dinh cua hoc sinh.
    if len(filled) < 2:
        return False
    means = [_patch(norm, _snap(score, question_bubble(q, c)))[_CORE].mean() for c in filled]
    return max(means) - min(means) > MAX_DARKNESS_GAP


def _small(img):
    scale = STEP_MAX_SIDE / max(img.shape[:2])
    return cv2.resize(img, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA) if scale < 1 else img


def read_sheet(path):
    """Doc mot anh bai lam; anh khong doc duoc thi ghi ly do vao problems, khong raise."""
    r = SheetReading()
    # Chi bat SheetUnreadable (loi "du doan duoc"): ghi ly do vao problems de grade_reading ra "Can xem lai".
    try:
        bgr = load_image(path)
        r.steps["goc"] = _small(bgr)
        gray, _, binary = preprocess(bgr)
        r.steps["xam"] = _small(gray)
        r.steps["nhi_phan"] = _small(binary)
        corners = find_markers(binary, gray)
        marked = bgr.copy()
        for x, y in corners:
            cv2.circle(marked, (int(x), int(y)), 40, (0, 0, 255), 8)
        r.steps["dinh_vi"] = _small(marked)
        warped = fix_orientation(warp(gray, corners))
    except SheetUnreadable as e:
        r.problems.append(e.reason)
        return r
    r.steps["nan_phoi_canh"] = _small(warped)
    warped = unbend(warped)  # buoc 5: bu giay cong
    r.warped = warped
    r.steps["nan_giay_cong"] = _small(warped)
    light = _normalize_light(warped)
    score = ring_score(light)
    norm, blank_ref = _blank_reference(light, score)
    r.steps["khu_anh_sang"] = _small((np.clip(norm, 0, 1) * 255).astype(np.uint8))
    r.problems += check_quality(warped, norm, score, blank_ref)
    r.student_id, r.id_problems = _read_id(norm, score, blank_ref)
    # Doc tat ca 40 cau (ke ca cau khong co trong dap an: grade_reading tu loc theo key).
    for q in range(1, NUM_QUESTIONS + 1):
        states = {c: _state(norm, score, question_bubble(q, c), blank_ref) for c in CHOICES}
        r.marks[q] = [c for c, s in states.items() if s == "filled"]
        # Cau "uncertain" neu co bat ky o nao uncertain, hoac 2 o to dam khong deu nhau.
        if "uncertain" in states.values() or _uneven_marks(norm, score, q, r.marks[q]):
            r.uncertain.append(q)
    return r

## Bước 5. Chấm điểm
Điểm = số câu đúng / N × 10. Câu trống hoặc tô nhiều ô được 0 điểm và có ghi chú. Bài có điểm không chắc chắn được đánh dấu **Cần xem lại** và không cho điểm: thà để giáo viên xem lại còn hơn chấm sai.

In [ ]:
"""Doc dap an, cham diem, ve anh danh dau va xuat CSV.

Day la tang QUYET DINH cuoi: omr_read chi bao "doc duoc gi, chac hay khong"; file nay quyet dinh
cho diem hay tra "Can xem lai". Quy tac xuyen suot: doc khong chac -> khong cho diem (slide 3 va 9).
"""
import csv
import os
from collections import Counter
from dataclasses import dataclass, field

import cv2
import numpy as np


DONE = "Đã chấm"
REVIEW = "Cần xem lại"
_GREEN, _RED = (0, 170, 0), (0, 0, 230)


class AnswerKeyError(Exception):
    pass


@dataclass
class SheetResult:
    filename: str
    status: str
    student_id: str | None = None
    correct: int | None = None
    score: float | None = None
    notes: list = field(default_factory=list)
    annotated: np.ndarray | None = None
    steps: dict = field(default_factory=dict)


def _trim_row(row):
    """Bo khoang trang va cac cot rong o cuoi dong."""
    cells = [c.strip() for c in row]
    while cells and not cells[-1]:
        cells.pop()
    return cells


def load_answer_key(path):
    """Doc CSV hai cot cau,dap_an; chap nhan file luu tu Excel (BOM, dau ;, chu thuong)."""
    try:
        with open(path, encoding="utf-8-sig", newline="") as f:
            lines = [ln for ln in f.read().splitlines() if ln.strip()]
    except (OSError, UnicodeDecodeError) as e:
        raise AnswerKeyError("Không đọc được file đáp án") from e
    if lines and lines[0].lower().startswith("sep="):
        lines = lines[1:]
    if not lines:
        raise AnswerKeyError("File đáp án trống")
    delim = ";" if ";" in lines[0] else ","
    rows = [_trim_row(row) for row in csv.reader(lines, delimiter=delim)]
    rows = [row for row in rows if row]  # Excel hay de lai dong toan dau phan cach
    if not rows or [c.lower() for c in rows[0]] != ["cau", "dap_an"]:
        raise AnswerKeyError("Dòng đầu của file đáp án phải là: cau,dap_an")
    key = {}
    for num, row in enumerate(rows[1:], start=2):
        if len(row) != 2 or not row[0].isdigit():
            raise AnswerKeyError(f"Dòng {num} sai định dạng")
        q, answer = int(row[0]), row[1].upper()
        if len(answer) != 1 or answer not in CHOICES:
            raise AnswerKeyError(f"Dòng {num}: đáp án phải là một trong A, B, C, D")
        if q != len(key) + 1:
            raise AnswerKeyError(f"Dòng {num}: số câu phải liên tục từ 1")
        key[q] = answer
    if not 1 <= len(key) <= NUM_QUESTIONS:
        raise AnswerKeyError(f"Đáp án phải có từ 1 đến {NUM_QUESTIONS} câu")
    return key


def _annotate(warped, marks, key):
    """Vong xanh o dap an dung, vong do o hoc sinh to sai."""
    out = cv2.cvtColor(warped, cv2.COLOR_GRAY2BGR)
    for q, answer in key.items():
        cv2.circle(out, question_bubble(q, answer), BUBBLE_RADIUS + 6, _GREEN, 4)
        for c in marks.get(q, []):
            if c != answer:
                cv2.circle(out, question_bubble(q, c), BUBBLE_RADIUS + 6, _RED, 4)
    return out


def grade_reading(filename, reading, key):
    """Bai doc khong chac chan thi "Can xem lai", khong bao gio cho diem."""
    # Chi tinh cac cau uncertain nam trong dap an (de thi 20 cau thi cau 21-40 khong lien quan).
    result = SheetResult(filename, DONE, reading.student_id, steps=reading.steps)
    uncertain = [q for q in reading.uncertain if q in key]
    notes = list(reading.problems) + list(reading.id_problems)
    if uncertain:
        notes.append("Câu không đọc chắc chắn: " + ", ".join(map(str, uncertain)))
    # Ve anh danh dau ca khi "Can xem lai", de giao vien nhin thay cho nghi ngo.
    if reading.warped is not None:
        result.annotated = _annotate(reading.warped, reading.marks, key)
    # Co BAT KY ghi chu loi nao (anh mo, ma hoc sinh, cau khong chac) -> dung, khong cho diem.
    # Day la nhanh "an toan truoc": thieu mot bai tot de giao vien xem, con hon cham sai mot bai.
    if notes:
        result.status, result.notes = REVIEW, notes
        return result
    # Toi day moi doc chac chan. Cau bo trong / to nhieu o la quyet dinh cua hoc sinh, khong phai
    # loi doc: van cham (0 diem cau do) va chi ghi chu de giao vien biet.
    blank = [q for q in key if not reading.marks.get(q)]
    multi = [q for q in key if len(reading.marks.get(q, [])) > 1]
    if blank:
        notes.append("Câu bỏ trống: " + ", ".join(map(str, blank)))
    if multi:
        notes.append("Câu tô nhiều ô: " + ", ".join(map(str, multi)))
    # Dung chi khi hoc sinh to DUNG MOT o va do la dap an: so sanh danh sach marks == [a] nen
    # to 2 o (ke ca co o dung) deu tinh sai; cau bo trong marks = [] cung sai.
    result.correct = sum(reading.marks.get(q) == [a] for q, a in key.items())
    # Chia cho so cau trong DAP AN (len(key)), khong phai 40: de 20 cau van ra thang 10.
    result.score = round(result.correct / len(key) * 10, 2)
    result.notes = notes
    return result


def grade_batch(paths, key):
    results = []
    for p in paths:  # tung anh xu ly doc lap; mot anh hong khong duoc lam hong ca lo
        name = os.path.basename(p)
        try:
            results.append(grade_reading(name, read_sheet(p), key))
        except Exception as e:  # mot file loi khong duoc lam mat ket qua cua ca lop
            results.append(SheetResult(name, REVIEW, notes=[f"Lỗi khi xử lý ảnh: {e}"]))
    # Sau khi doc het moi biet trung ma. Chi ghi chu, khong tu y doi trang thai: co the la 2 hoc sinh
    # to trung ma, hoac mot hoc sinh nop 2 lan - giao vien quyet dinh.
    counts = Counter(r.student_id for r in results if r.student_id)
    for r in results:
        if counts.get(r.student_id, 0) > 1:
            r.notes.append(f"Trùng mã học sinh {r.student_id}")
    return results


def write_csv(results, path):
    """utf-8-sig de Excel hien dung tieng Viet (BOM bao Excel doc UTF-8 thay vi code page may)."""
    with open(path, "w", encoding="utf-8-sig", newline="") as f:
        w = csv.writer(f)
        w.writerow(["ten_file", "ma_hoc_sinh", "so_cau_dung", "diem", "trang_thai", "ghi_chu"])
        for r in results:
            w.writerow([r.filename, r.student_id or "",
                        "" if r.correct is None else r.correct,
                        "" if r.score is None else r.score,
                        r.status, "; ".join(r.notes)])

## Bước 6. Sinh ảnh mô phỏng
Tạo ảnh có xoay, nghiêng, tối và ánh sáng không đều với đáp án biết trước, dùng để thử hệ thống khi chưa có phiếu thật.

In [ ]:
"""Sinh anh mo phong anh chup phieu, da biet truoc dap an."""
import math

import cv2
import numpy as np



def fill_bubble(img, center, darkness=40, coverage=1.0):
    """To mot o; coverage < 1 chi to phan tren cua o, mo phong net to thieu."""
    x, y = center
    r = int(BUBBLE_RADIUS * 0.9)
    mask = np.zeros(img.shape, np.uint8)
    cv2.circle(mask, (x, y), r, 255, -1)
    mask[y - r + int(2 * r * coverage):] = 0
    img[mask > 0] = darkness


def fill_sheet(img, student_id, answers):
    """Tra ve ban sao cua phieu da to ma hoc sinh va cac cau tra loi."""
    out = img.copy()
    for pos, ch in enumerate(student_id):
        fill_bubble(out, id_bubble(pos, int(ch)))
    for q, choice in answers.items():
        fill_bubble(out, question_bubble(q, choice))
    return out


def photograph(img, angle=0.0, perspective=0.0, brightness=1.0, gradient=0.0, blur=0,
               background=200, seed=0):
    """Mo phong anh chup: dat phieu len nen, xoay, nghieng, doi sang, lam mo, them nhieu."""
    rng = np.random.default_rng(seed)
    side = int(math.hypot(PAGE_W, PAGE_H)) + 200
    cx, cy = side / 2, side / 2
    a = math.radians(angle)
    corners = []
    for sx, sy in [(-1, -1), (1, -1), (1, 1), (-1, 1)]:
        dx = sx * PAGE_W / 2 + rng.uniform(-perspective, perspective) * PAGE_W
        dy = sy * PAGE_H / 2 + rng.uniform(-perspective, perspective) * PAGE_W
        corners.append((cx + dx * math.cos(a) - dy * math.sin(a),
                        cy + dx * math.sin(a) + dy * math.cos(a)))
    src = np.float32([[0, 0], [PAGE_W, 0], [PAGE_W, PAGE_H], [0, PAGE_H]])
    m = cv2.getPerspectiveTransform(src, np.float32(corners))
    out = cv2.warpPerspective(img, m, (side, side), flags=cv2.INTER_LINEAR,
                              borderValue=int(background)).astype(np.float32)
    out *= brightness
    if gradient:
        out *= np.linspace(1.0, 1.0 - gradient, side, dtype=np.float32)[None, :]
    if blur:
        k = blur | 1
        out = cv2.GaussianBlur(out, (k, k), 0)
    out += rng.normal(0, 2.5, out.shape).astype(np.float32)
    return cv2.cvtColor(np.clip(out, 0, 255).astype(np.uint8), cv2.COLOR_GRAY2BGR)


def add_glare(img):
    """Phu mot vung trang loa len khu vuc o tra loi cua phieu xam."""
    out = img.copy()
    x, y = question_bubble(11, "A")
    cv2.circle(out, (x + 100, y + 300), 320, 255, -1)
    return out


def bend(img, dx=30, dy=40, center=(250, 1000), sigma=300):
    """Lam cong giay quanh mot diem (nhu cho tay cam): noi dung lech toi (dx, dy), o goc gan nhu khong lech."""
    ys, xs = np.mgrid[0:img.shape[0], 0:img.shape[1]].astype(np.float32)
    w = np.exp(-((xs - center[0]) ** 2 + (ys - center[1]) ** 2) / (2 * sigma ** 2))
    return cv2.remap(img, xs - dx * w, ys - dy * w, cv2.INTER_LINEAR, borderValue=255)


def move_bar(img, dy=-22):
    """Dich thanh dinh huong theo chieu doc, mo phong giay cong o mep tren."""
    out = img.copy()
    x, y, w, h = ORIENTATION_BAR
    out[y:y + h, x:x + w] = 255
    out[y + dy:y + dy + h, x:x + w] = 0
    return out


def add_dark_keys(bgr, count=14, size=45, key=60, base=140):
    """Phu nen kieu ban phim: nhieu phim xam toi, lon hon o dinh vi, tren nen chi sang hon mot chut."""
    out = bgr.copy()
    out[:, :120] = base
    out[:120, :] = base
    for i in range(count):
        x = 10 + (i % 7) * 110
        out[20:20 + size, 120 + x:120 + x + size] = key
        out[200 + (i // 7) * 110:200 + (i // 7) * 110 + size, 20:20 + size] = key
    return out


def shift_content(img, dx=25):
    """Dich noi dung so voi o dinh vi, mo phong luoi o bi lech."""
    out = img.copy()
    box = out[170:2200, 160:1500].copy()
    out[170:2200, 160:1500] = 255
    out[170:2200, 160 + dx:1500 + dx] = box
    return out

## Bước 7. Giao diện Gradio

In [ ]:
"""Giao dien Gradio: upload dap an va bai lam, xem diem va tung buoc xu ly."""
import os
import tempfile

import cv2
import gradio as gr


COLUMNS = ["Tên file", "Mã học sinh", "Số câu đúng", "Điểm", "Trạng thái", "Ghi chú"]


def _rgb(img):
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB) if img.ndim == 3 else img


def run_grading(key_path, image_paths):
    """Tra ve (hang cua bang, duong dan CSV, ket qua, thong bao)."""
    if not key_path:
        return [], None, [], "Hãy upload file đáp án."
    if not image_paths:
        return [], None, [], "Hãy upload ít nhất một ảnh bài làm."
    try:
        key = load_answer_key(key_path)
    except AnswerKeyError as e:
        return [], None, [], f"Lỗi file đáp án: {e}"
    results = grade_batch(image_paths, key)
    rows = [[r.filename, r.student_id or "", r.correct, r.score, r.status, "; ".join(r.notes)]
            for r in results]
    csv_path = os.path.join(tempfile.mkdtemp(), "scores.csv")
    write_csv(results, csv_path)
    review = sum(r.score is None for r in results)
    return rows, csv_path, results, f"Đã xử lý {len(results)} bài, {review} bài cần xem lại."


def show_sheet(index, results):
    """Tra ve (anh danh dau, [(anh tung buoc, ten buoc)]) cua bai thu index."""
    if index is None or not results or index >= len(results):
        return None, []
    r = results[index]
    annotated = None if r.annotated is None else _rgb(r.annotated)
    return annotated, [(_rgb(img), step) for step, img in r.steps.items()]


def build_ui():
    with gr.Blocks(title="Chấm bài trắc nghiệm") as demo:
        gr.Markdown("# Hệ thống chấm bài trắc nghiệm")
        results = gr.State([])
        with gr.Row():
            key = gr.File(label="1. File đáp án (CSV: cau,dap_an)", file_count="single",
                          file_types=[".csv"])
            images = gr.File(label="2. Ảnh bài làm (chọn được nhiều file)", file_count="multiple",
                             file_types=["image"])
        run = gr.Button("Chấm bài", variant="primary")
        message = gr.Markdown()
        table = gr.Dataframe(headers=COLUMNS, datatype="str", label="Kết quả", interactive=False)
        csv_file = gr.File(label="Tải file điểm (Excel có thể ẩn số 0 đầu mã học sinh)", interactive=False)
        choice = gr.Dropdown(label="Xem chi tiết bài", choices=[])
        with gr.Row():
            annotated = gr.Image(label="Đánh dấu: xanh là đáp án đúng, đỏ là ô tô sai")
            steps = gr.Gallery(label="Các bước xử lý ảnh", columns=3)

        def on_run(key_path, image_paths):
            rows, csv_path, res, msg = run_grading(key_path, image_paths)
            names = [(f"{i + 1}. {r.filename}", i) for i, r in enumerate(res)]
            return (rows, csv_path, res, msg, gr.update(choices=names, value=None), None, [])

        run.click(on_run, [key, images], [table, csv_file, results, message, choice, annotated, steps])
        choice.change(show_sheet, [choice, results], [annotated, steps])
    return demo

## Bước 8. Tạo phiếu, dữ liệu mẫu và thử một bài

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt

Path("templates").mkdir(exist_ok=True)
save_pdf("templates/answer_sheet.pdf")

mau = Path("samples")
mau.mkdir(exist_ok=True)
dap_an = {q: "ABCD"[q % 4] for q in range(1, 41)}
(mau / "sample_answer_key.csv").write_text(
    "cau,dap_an\n" + "".join(f"{q},{a}\n" for q, a in dap_an.items()))

bai_lam = dict(dap_an)
bai_lam[3] = "A" if dap_an[3] != "A" else "B"  # co y to sai cau 3
del bai_lam[7]  # co y bo trong cau 7
phieu = fill_sheet(render_sheet(), "001234", bai_lam)
anh_mau = mau / "sample_submission.jpg"
cv2.imwrite(str(anh_mau), photograph(phieu, angle=6, perspective=.03, gradient=.4))

ket_qua = grade_batch([str(anh_mau)], load_answer_key(str(mau / "sample_answer_key.csv")))[0]
print(ket_qua.status, ket_qua.student_id, ket_qua.score, ket_qua.notes)

fig, axes = plt.subplots(2, 4, figsize=(18, 10))
for ax in axes.flat:
    ax.axis("off")
for ax, (ten, anh) in zip(axes.flat, ket_qua.steps.items()):
    ax.imshow(anh[..., ::-1] if anh.ndim == 3 else anh, cmap="gray")
    ax.set_title(ten)
plt.show()

plt.figure(figsize=(8, 11))
plt.imshow(ket_qua.annotated[..., ::-1])
plt.axis("off")
plt.show()

## Bước 9. Mở giao diện chấm bài

In [ ]:
import os

demo = build_ui()
if not os.environ.get("OMR_SKIP_LAUNCH"):
    # share=False: giao dien hien ngay trong Colab, khong tao link cong khai chua bai cua hoc sinh.
    # Neu khung giao dien khong hien, doi thanh share=True (bat ky ai co link deu vao duoc).
    demo.launch(share=False)